In [1]:
import darts
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

In [2]:
import csv

input_file = "data.csv"
output_file = "single_device.csv"

device_id_to_keep = None

with open(input_file, newline="") as infile, open(output_file, "w", newline="") as outfile:
    reader = csv.DictReader(infile)
    writer = csv.DictWriter(outfile, fieldnames=reader.fieldnames)

    writer.writeheader()

    for row in reader:
        if device_id_to_keep is None:
            device_id_to_keep = row["deviceId"]
            print("Selected device:", device_id_to_keep)

        if row["deviceId"] == device_id_to_keep:
            writer.writerow(row)

Selected device: 000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc93ae46eecc5798f0fe


In [5]:
df = pd.read_csv('single_device.csv')

df["timedate"] = pd.to_datetime(df["timedate"], utc=True)

# encode time features
df["month"] = df["timedate"].dt.month
df["hour"] = df["timedate"].dt.hour

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df.drop(columns=["deviceId", "timedate", "hour", "month"], inplace=True)

# split datasets
train_df = df[df["period"] == "train"].copy()
valid_df = df[df["period"] == "valid"].copy()
test_df  = df[df["period"] == "test"].copy()

train_df.drop(columns=['period'], inplace=True)
valid_df.drop(columns=['period'], inplace=True)
test_df.drop(columns=['period'], inplace=True)

In [6]:
train_df.head()

,t1,t2,t3,t4,t5,t6,t7,t8,t9,t10,...,t12,t13,x1,x2,x3,deviceType,month_sin,month_cos,hour_sin,hour_cos
0,0.29,0.05,0.0,0.43,0.47,0.45,0.2,0.51,0.42,0.21,...,0.07,0.07,0.0,0.0,8,19,-0.866025,0.5,0.0,1.0
1,0.29,0.05,0.0,0.39,0.46,0.45,0.2,0.51,0.41,0.21,...,0.07,0.07,0.0,0.0,8,19,-0.866025,0.5,0.0,1.0
2,0.29,0.05,0.0,0.38,0.46,0.45,0.2,0.50,0.41,0.21,...,0.07,0.07,0.0,0.0,8,19,-0.866025,0.5,0.0,1.0
3,0.29,0.05,0.0,0.38,0.45,0.45,0.2,0.50,0.40,0.21,...,0.07,0.07,0.0,0.0,8,19,-0.866025,0.5,0.0,1.0
4,0.29,0.05,0.0,0.37,0.45,0.45,0.2,0.50,0.40,0.21,...,0.07,0.07,0.0,0.0,8,19,-0.866025,0.5,0.0,1.0


In [7]:
train_df.shape

(60789, 21)

In [8]:
split_point = int(0.8 * 60789)
train_test_df = train_df[:split_point]
train_valid_df = train_df[split_point:]

In [18]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

target = "x2"

X = train_df.drop(columns=[target])
y = train_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert to DMatrix (optimized structure for XGBoost)
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# Model parameters
params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "max_depth": 6,
    "eta": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
}

# Train model
model = xgb.train(
    params,
    dtrain,
    num_boost_round=200,
    evals=[(dtrain, "train"), (dtest, "test")],
    early_stopping_rounds=20,
)

# Predict
preds = model.predict(dtest)

# Evaluate
rmse = mean_absolute_error(y_test, preds)
print("RMSE:", rmse)

[0]	train-rmse:0.18238	test-rmse:0.18217
[1]	train-rmse:0.16857	test-rmse:0.16824
[2]	train-rmse:0.15514	test-rmse:0.15480
[3]	train-rmse:0.14395	test-rmse:0.14362
[4]	train-rmse:0.13325	test-rmse:0.13300
[5]	train-rmse:0.12365	test-rmse:0.12345
[6]	train-rmse:0.11525	test-rmse:0.11508
[7]	train-rmse:0.10786	test-rmse:0.10782
[8]	train-rmse:0.10130	test-rmse:0.10135
[9]	train-rmse:0.09536	test-rmse:0.09539
[10]	train-rmse:0.09045	test-rmse:0.09051
[11]	train-rmse:0.08610	test-rmse:0.08619
[12]	train-rmse:0.08246	test-rmse:0.08253
[13]	train-rmse:0.07909	test-rmse:0.07922
[14]	train-rmse:0.07617	test-rmse:0.07638
[15]	train-rmse:0.07374	test-rmse:0.07400
[16]	train-rmse:0.07159	test-rmse:0.07189
[17]	train-rmse:0.06979	test-rmse:0.07007
[18]	train-rmse:0.06785	test-rmse:0.06817
[19]	train-rmse:0.06647	test-rmse:0.06680
[20]	train-rmse:0.06542	test-rmse:0.06574
[21]	train-rmse:0.06434	test-rmse:0.06467
[22]	train-rmse:0.06329	test-rmse:0.06362
[23]	train-rmse:0.06245	test-rmse:0.06280
[2

In [17]:
train_df.describe()

,t1,t2,t3,t4,t5,t6,t7,t8,t9,t10,...,t12,t13,x1,x2,x3,deviceType,month_sin,month_cos,hour_sin,hour_cos
count,60789.000000,60789.000000,60789.0,60789.000000,60789.000000,60789.000000,60789.000000,60789.000000,60789.000000,60789.000000,...,60789.000000,60789.000000,60789.0,60789.000000,60789.0,60789.0,60789.000000,6.078900e+04,60789.000000,6.078900e+04
mean,0.283824,0.042482,0.0,0.422744,0.463730,0.448843,0.206733,0.519112,0.403024,0.207171,...,0.068619,0.071984,0.0,0.187953,8.0,19.0,0.258426,4.624654e-01,-0.000803,2.731307e-03
std,0.028845,0.004319,0.0,0.056188,0.033966,0.029595,0.004717,0.019955,0.031974,0.005689,...,0.004641,0.006505,0.0,0.198773,0.0,0.0,0.683479,5.022044e-01,0.707405,7.068140e-01
min,0.220000,0.040000,0.0,0.270000,0.330000,0.330000,0.180000,0.450000,0.320000,0.200000,...,0.050000,0.050000,0.0,0.000000,8.0,19.0,-0.866025,-5.000000e-01,-1.000000,-1.000000e+00
25%,0.260000,0.040000,0.0,0.370000,0.440000,0.430000,0.200000,0.500000,0.380000,0.200000,...,0.070000,0.070000,0.0,0.000000,8.0,19.0,-0.500000,6.123234e-17,-0.707107,-7.071068e-01
50%,0.280000,0.040000,0.0,0.430000,0.470000,0.450000,0.210000,0.520000,0.410000,0.210000,...,0.070000,0.070000,0.0,0.245963,8.0,19.0,0.500000,5.000000e-01,0.000000,6.123234e-17
75%,0.300000,0.040000,0.0,0.480000,0.490000,0.470000,0.210000,0.530000,0.430000,0.210000,...,0.070000,0.080000,0.0,0.317371,8.0,19.0,0.866025,8.660254e-01,0.707107,7.071068e-01
max,0.380000,0.050000,0.0,0.520000,0.550000,0.520000,0.210000,0.600000,0.450000,0.230000,...,0.080000,0.090000,0.0,0.666480,8.0,19.0,1.000000,1.000000e+00,1.000000,1.000000e+00
